In [1]:
import duckdb
con = duckdb.connect('../data/cfpb.duckdb')

In [2]:
con.execute("""
CREATE OR REPLACE TABLE complaints_raw AS
SELECT * FROM read_csv_auto('../data/raw/complaints_structured.csv',
                            header=true, all_varchar=true)
""")
con.sql("DESCRIBE complaints_raw").show()

┌──────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│         column_name          │ column_type │  null   │   key   │ default │  extra  │
│           varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Date received                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Product                      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Sub-product                  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Issue                        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Sub-issue                    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Company public response      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Company                      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ State                        │ VARCHAR   

In [6]:
con.execute("""
CREATE OR REPLACE TABLE complaints AS
SELECT
  CAST("Complaint ID" AS BIGINT) AS complaint_id,
  CAST(COALESCE(TRY_STRPTIME("Date received", '%m/%d/%Y'),
                TRY_STRPTIME("Date received", '%Y-%m-%d')) AS DATE) AS date_received,
  "Product" AS product,
  "Sub-product" AS sub_product,
  "Issue" AS issue,
  "Sub-issue" AS sub_issue,
  "Company public response" AS company_public_response,
  "Company" AS company,
  "State" AS state,
  "ZIP code" AS zip_code,
  "Tags" AS tags,
  "Submitted via" AS submitted_via,
  "Company response to consumer" AS company_response,
  "Timely response?" AS timely_response
FROM complaints_raw
WHERE CAST(COALESCE(TRY_STRPTIME("Date received", '%m/%d/%Y'),
                    TRY_STRPTIME("Date received", '%Y-%m-%d')) AS DATE)
      BETWEEN DATE '2023-09-01' AND DATE '2026-08-14'
""")
con.sql("SELECT COUNT(*) AS rows FROM complaints").show()

┌────────┐
│  rows  │
│ int64  │
├────────┤
│ 605225 │
└────────┘



In [5]:
print(con.sql("SELECT * FROM complaints_raw LIMIT 0").columns)

['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Complaint ID']


In [7]:
arch = con.sql("""
SELECT * FROM read_csv_auto('../data/raw/archive/*.csv', all_varchar=true, union_by_name=true) LIMIT 0
""")
print(arch.columns)

['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Complaint ID']


In [8]:
con.sql("""
SELECT COUNT(*) AS rows,
       COUNT(NULLIF("Consumer complaint narrative", '')) AS rows_with_narrative
FROM read_csv_auto('../data/raw/archive/*.csv', all_varchar=true, union_by_name=true)
""").show()

┌─────────┬─────────────────────┐
│  rows   │ rows_with_narrative │
│  int64  │        int64        │
├─────────┼─────────────────────┤
│ 1339176 │               10003 │
└─────────┴─────────────────────┘



In [9]:
con.sql("""
SELECT date_trunc('month', COALESCE(TRY_STRPTIME("Date received",'%m/%d/%Y'),
                                    TRY_STRPTIME("Date received",'%Y-%m-%d'))) AS month,
       COUNT(*) AS rows,
       COUNT(NULLIF("Consumer complaint narrative",'')) AS with_narrative,
       ROUND(100.0*COUNT(NULLIF("Consumer complaint narrative",''))/COUNT(*),2) AS pct
FROM read_csv_auto('../data/raw/archive/*.csv', all_varchar=true, union_by_name=true)
GROUP BY 1 ORDER BY 1
""").show()

┌─────────────────────┬────────┬────────────────┬────────┐
│        month        │  rows  │ with_narrative │  pct   │
│      timestamp      │ int64  │     int64      │ double │
├─────────────────────┼────────┼────────────────┼────────┤
│ 2026-07-01 00:00:00 │ 672036 │          10002 │   1.49 │
│ 2026-08-01 00:00:00 │ 667140 │              1 │    0.0 │
└─────────────────────┴────────┴────────────────┴────────┘



In [10]:
con.sql("""
SELECT COUNT(*) AS in_scope_with_narrative
FROM read_csv_auto('../data/raw/archive/*.csv', all_varchar=true, union_by_name=true) a
JOIN complaints c ON CAST(a."Complaint ID" AS BIGINT) = c.complaint_id
WHERE NULLIF(a."Consumer complaint narrative",'') IS NOT NULL
""").show()

┌─────────────────────────┐
│ in_scope_with_narrative │
│          int64          │
├─────────────────────────┤
│                    4762 │
└─────────────────────────┘



In [11]:
con.sql("""
SELECT date_trunc('month', c.date_received) AS month,
       COUNT(*) AS in_scope_rows,
       COUNT(NULLIF(a."Consumer complaint narrative",'')) AS with_narrative,
       ROUND(100.0*COUNT(NULLIF(a."Consumer complaint narrative",''))/COUNT(*),1) AS pct
FROM complaints c
LEFT JOIN read_csv_auto('../data/raw/archive/*.csv', all_varchar=true, union_by_name=true) a
  ON CAST(a."Complaint ID" AS BIGINT) = c.complaint_id
GROUP BY 1 ORDER BY 1
""").show()

┌─────────────────────┬───────────────┬────────────────┬────────┐
│        month        │ in_scope_rows │ with_narrative │  pct   │
│      timestamp      │     int64     │     int64      │ double │
├─────────────────────┼───────────────┼────────────────┼────────┤
│ 2023-09-01 00:00:00 │         10333 │           6352 │   61.5 │
│ 2023-10-01 00:00:00 │         11422 │           6683 │   58.5 │
│ 2023-11-01 00:00:00 │         10512 │           5971 │   56.8 │
│ 2023-12-01 00:00:00 │         10524 │           6080 │   57.8 │
│ 2024-01-01 00:00:00 │         11246 │           6322 │   56.2 │
│ 2024-02-01 00:00:00 │         10874 │           6110 │   56.2 │
│ 2024-03-01 00:00:00 │         12200 │           7053 │   57.8 │
│ 2024-04-01 00:00:00 │         12841 │              0 │    0.0 │
│ 2024-05-01 00:00:00 │         13185 │              0 │    0.0 │
│ 2024-06-01 00:00:00 │         11183 │              0 │    0.0 │
│          ·          │           ·   │              · │     ·  │
│         

In [12]:
con.sql("""
SELECT COUNT(*) AS rows, MIN(date_received) AS first_date, MAX(date_received) AS last_date
FROM complaints
""").show()

┌────────┬────────────┬────────────┐
│  rows  │ first_date │ last_date  │
│ int64  │    date    │    date    │
├────────┼────────────┼────────────┤
│ 605225 │ 2023-09-01 │ 2026-08-14 │
└────────┴────────────┴────────────┘



In [13]:
con.sql("""
SELECT COUNT(*) - COUNT(DISTINCT complaint_id) AS duplicate_ids FROM complaints
""").show()

┌───────────────┐
│ duplicate_ids │
│     int64     │
├───────────────┤
│             0 │
└───────────────┘



In [14]:
con.sql("""
SELECT product, COUNT(*) AS complaints FROM complaints GROUP BY 1 ORDER BY 2 DESC
""").show()

┌────────────────────────────────────────────────────┬────────────┐
│                      product                       │ complaints │
│                      varchar                       │   int64    │
├────────────────────────────────────────────────────┼────────────┤
│ Credit card                                        │     247292 │
│ Checking or savings account                        │     208577 │
│ Money transfer, virtual currency, or money service │     130388 │
│ Prepaid card                                       │      18968 │
└────────────────────────────────────────────────────┴────────────┘



In [15]:
con.sql("""
SELECT company, COUNT(*) AS complaints FROM complaints GROUP BY 1 ORDER BY 2 DESC LIMIT 20
""").show()

┌────────────────────────────────────────┬────────────┐
│                company                 │ complaints │
│                varchar                 │   int64    │
├────────────────────────────────────────┼────────────┤
│ Block, Inc.                            │      61656 │
│ JPMORGAN CHASE & CO.                   │      46741 │
│ CAPITAL ONE FINANCIAL CORPORATION      │      42133 │
│ BANK OF AMERICA, NATIONAL ASSOCIATION  │      36764 │
│ WELLS FARGO & COMPANY                  │      35367 │
│ CITIBANK, N.A.                         │      35337 │
│ Early Warning Services, LLC            │      22503 │
│ NAVY FEDERAL CREDIT UNION              │      20099 │
│ SYNCHRONY FINANCIAL                    │      20043 │
│ Paypal Holdings, Inc                   │      19965 │
│ Chime Financial Inc                    │      19848 │
│ AMERICAN EXPRESS COMPANY               │      19239 │
│ EQUIFAX, INC.                          │      18660 │
│ TRANSUNION INTERMEDIATE HOLDINGS, INC. │      

In [16]:
con.sql("""
SELECT company, COUNT(*) AS complaints FROM complaints
WHERE company ILIKE '%chime%' OR company ILIKE '%block%' OR company ILIKE '%paypal%'
   OR company ILIKE '%sofi%' OR company ILIKE '%varo%' OR company ILIKE '%dave%'
   OR company ILIKE '%moneylion%' OR company ILIKE '%robinhood%' OR company ILIKE '%coinbase%'
   OR company ILIKE '%wise%' OR company ILIKE '%remitly%'
GROUP BY 1 ORDER BY 2 DESC
""").show(max_rows=50)

┌─────────────────────────┬────────────┐
│         company         │ complaints │
│         varchar         │   int64    │
├─────────────────────────┼────────────┤
│ Block, Inc.             │      61656 │
│ Paypal Holdings, Inc    │      19965 │
│ Chime Financial Inc     │      19848 │
│ SOFI TECHNOLOGIES, INC. │       5470 │
│ ROBINHOOD MARKETS INC.  │       3936 │
│ Coinbase, Inc.          │       3292 │
│ TransferWise Ltd        │       1390 │
│ Remitly, Inc.           │        591 │
│ MoneyLion Inc.          │        510 │
│ Dave Operating, LLC     │        355 │
│ Blockchain.com, Inc.    │        255 │
│ Chime Inc.              │        216 │
│ SOFI LOCAL CORP         │          1 │
└─────────────────────────┴────────────┘
  13 rows                    2 columns



In [17]:
con.sql("""
SELECT date_trunc('month', date_received) AS month, COUNT(*) AS complaints
FROM complaints GROUP BY 1 ORDER BY 1
""").show(max_rows=40)

┌─────────────────────┬────────────┐
│        month        │ complaints │
│      timestamp      │   int64    │
├─────────────────────┼────────────┤
│ 2023-09-01 00:00:00 │      10333 │
│ 2023-10-01 00:00:00 │      11422 │
│ 2023-11-01 00:00:00 │      10512 │
│ 2023-12-01 00:00:00 │      10524 │
│ 2024-01-01 00:00:00 │      11246 │
│ 2024-02-01 00:00:00 │      10874 │
│ 2024-03-01 00:00:00 │      12200 │
│ 2024-04-01 00:00:00 │      12841 │
│ 2024-05-01 00:00:00 │      13185 │
│ 2024-06-01 00:00:00 │      11183 │
│ 2024-07-01 00:00:00 │      12327 │
│ 2024-08-01 00:00:00 │      14053 │
│ 2024-09-01 00:00:00 │      13120 │
│ 2024-10-01 00:00:00 │      13765 │
│ 2024-11-01 00:00:00 │      13144 │
│ 2024-12-01 00:00:00 │      13946 │
│ 2025-01-01 00:00:00 │      77939 │
│ 2025-02-01 00:00:00 │      18288 │
│ 2025-03-01 00:00:00 │      14842 │
│ 2025-04-01 00:00:00 │      15138 │
│ 2025-05-01 00:00:00 │      16074 │
│ 2025-06-01 00:00:00 │      15819 │
│ 2025-07-01 00:00:00 │      17891 │
│

In [21]:
con.sql("""
SELECT company,
       COUNT(*) FILTER (WHERE date_received BETWEEN DATE '2024-12-01' AND DATE '2024-12-31') AS dec_2024,
       COUNT(*) FILTER (WHERE date_received BETWEEN DATE '2025-01-01' AND DATE '2025-01-31') AS jan_2025,
       COUNT(*) FILTER (WHERE date_received BETWEEN DATE '2025-02-01' AND DATE '2025-02-28') AS feb_2025
FROM complaints
WHERE date_received BETWEEN DATE '2024-12-01' AND DATE '2025-02-28'
GROUP BY 1 ORDER BY jan_2025 - dec_2024 DESC LIMIT 10;
""").show()

┌────────────────────────────────────────┬──────────┬──────────┬──────────┐
│                company                 │ dec_2024 │ jan_2025 │ feb_2025 │
│                varchar                 │  int64   │  int64   │  int64   │
├────────────────────────────────────────┼──────────┼──────────┼──────────┤
│ Block, Inc.                            │      358 │    31223 │     3825 │
│ Early Warning Services, LLC            │      116 │    16723 │     1354 │
│ NAVY FEDERAL CREDIT UNION              │      417 │     7697 │      848 │
│ CAPITAL ONE FINANCIAL CORPORATION      │      803 │     5849 │     1037 │
│ JPMORGAN CHASE & CO.                   │     1186 │     1934 │     1168 │
│ BANK OF AMERICA, NATIONAL ASSOCIATION  │      850 │     1356 │      900 │
│ WELLS FARGO & COMPANY                  │      907 │     1362 │      875 │
│ TRANSUNION INTERMEDIATE HOLDINGS, INC. │      664 │      999 │      551 │
│ EQUIFAX, INC.                          │      643 │      934 │      528 │
│ Experian I

In [20]:
con.close()
con = duckdb.connect('../data/cfpb.duckdb')
con.sql("SELECT COUNT(*) AS rows FROM complaints").show()

┌────────┐
│  rows  │
│ int64  │
├────────┤
│ 605225 │
└────────┘



In [22]:
con.sql("""
SELECT date_received, COUNT(*) AS complaints FROM complaints
WHERE date_received BETWEEN DATE '2025-01-01' AND DATE '2025-01-31'
GROUP BY 1 ORDER BY 2 DESC LIMIT 5;
""").show()

┌───────────────┬────────────┐
│ date_received │ complaints │
│     date      │   int64    │
├───────────────┼────────────┤
│ 2025-01-17    │      19116 │
│ 2025-01-18    │      11732 │
│ 2025-01-16    │       7233 │
│ 2025-01-15    │       6532 │
│ 2025-01-20    │       5355 │
└───────────────┴────────────┘



In [23]:
con.sql("""
SELECT product, issue, COUNT(*) AS complaints FROM complaints
WHERE date_received BETWEEN DATE '2025-01-01' AND DATE '2025-01-31'
GROUP BY 1, 2 ORDER BY 3 DESC LIMIT 10;
""").show()

┌────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────┬────────────┐
│                      product                       │                              issue                              │ complaints │
│                      varchar                       │                             varchar                             │   int64    │
├────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────────┼────────────┤
│ Money transfer, virtual currency, or money service │ Other transaction problem                                       │      42521 │
│ Checking or savings account                        │ Managing an account                                             │       8232 │
│ Checking or savings account                        │ Problem caused by your funds being low                          │       7570 │
│ Money transfer, virtual currency, or money service │ Fraud o

In [24]:
con.sql("""
SELECT company, product, COUNT(*) AS complaints FROM complaints
WHERE company IN ('EQUIFAX, INC.', 'TRANSUNION INTERMEDIATE HOLDINGS, INC.', 'Experian Information Solutions Inc.')
GROUP BY 1, 2 ORDER BY 1, 3 DESC;
""").show()

┌────────────────────────────────────────┬────────────────────────────────────────────────────┬────────────┐
│                company                 │                      product                       │ complaints │
│                varchar                 │                      varchar                       │   int64    │
├────────────────────────────────────────┼────────────────────────────────────────────────────┼────────────┤
│ EQUIFAX, INC.                          │ Credit card                                        │      18318 │
│ EQUIFAX, INC.                          │ Checking or savings account                        │        187 │
│ EQUIFAX, INC.                          │ Prepaid card                                       │        116 │
│ EQUIFAX, INC.                          │ Money transfer, virtual currency, or money service │         39 │
│ Experian Information Solutions Inc.    │ Credit card                                        │      16077 │
│ Experian Informat

In [25]:
con.execute("""
CREATE OR REPLACE TABLE company_groups AS
SELECT * FROM (VALUES
  ('Block, Inc.', 'Fintech'),
  ('Paypal Holdings, Inc', 'Fintech'),
  ('Chime Financial Inc', 'Fintech'),
  ('Chime Inc.', 'Fintech'),
  ('SOFI TECHNOLOGIES, INC.', 'Fintech'),
  ('ROBINHOOD MARKETS INC.', 'Fintech'),
  ('Coinbase, Inc.', 'Fintech'),
  ('TransferWise Ltd', 'Fintech'),
  ('Remitly, Inc.', 'Fintech'),
  ('MoneyLion Inc.', 'Fintech'),
  ('Dave Operating, LLC', 'Fintech'),
  ('JPMORGAN CHASE & CO.', 'Bank'),
  ('BANK OF AMERICA, NATIONAL ASSOCIATION', 'Bank'),
  ('WELLS FARGO & COMPANY', 'Bank'),
  ('CAPITAL ONE FINANCIAL CORPORATION', 'Bank'),
  ('CITIBANK, N.A.', 'Bank')
) AS t(company, company_group)
""")
con.sql("""
SELECT g.company_group, COUNT(*) AS complaints
FROM complaints c JOIN company_groups g USING (company) GROUP BY 1
""").show()

┌───────────────┬────────────┐
│ company_group │ complaints │
│    varchar    │   int64    │
├───────────────┼────────────┤
│ Fintech       │     117229 │
│ Bank          │     196342 │
└───────────────┴────────────┘



In [26]:
con.sql("""
SELECT company, issue, COUNT(*) AS complaints
FROM complaints
WHERE date_received BETWEEN DATE '2025-01-15' AND DATE '2025-01-20'
GROUP BY 1, 2 ORDER BY 3 DESC LIMIT 10
""").show()

┌───────────────────────────────────┬────────────────────────────────────────────────────────┬────────────┐
│              company              │                         issue                          │ complaints │
│              varchar              │                        varchar                         │   int64    │
├───────────────────────────────────┼────────────────────────────────────────────────────────┼────────────┤
│ Block, Inc.                       │ Other transaction problem                              │      23424 │
│ Early Warning Services, LLC       │ Other transaction problem                              │      12374 │
│ NAVY FEDERAL CREDIT UNION         │ Problem caused by your funds being low                 │       4970 │
│ CAPITAL ONE FINANCIAL CORPORATION │ Managing an account                                    │       4132 │
│ Block, Inc.                       │ Other service problem                                  │        641 │
│ Early Warning Services, LL

In [27]:
con.sql("""
SELECT issue, COUNT(*) AS complaints
FROM complaints
WHERE issue ILIKE '%report%' OR issue ILIKE '%credit score%'
   OR issue ILIKE '%investigation%' OR issue ILIKE '%fraud alert%'
   OR issue ILIKE '%identity theft%' OR issue ILIKE '%monitoring%'
GROUP BY 1 ORDER BY 2 DESC
""").show(max_rows=30)

┌─────────────────────────────────────────────────────────────────┬────────────┐
│                              issue                              │ complaints │
│                             varchar                             │   int64    │
├─────────────────────────────────────────────────────────────────┼────────────┤
│ Incorrect information on your report                            │      32728 │
│ Problem with a company's investigation into an existing problem │      26519 │
│ Improper use of your report                                     │       2441 │
│ Credit monitoring or identity theft protection services         │        640 │
│ Problem with fraud alerts or security freezes                   │        361 │
│ Unable to get your credit report or credit score                │        145 │
└─────────────────────────────────────────────────────────────────┴────────────┘



In [31]:
con.execute("""
CREATE OR REPLACE TABLE complaints_flagged AS
SELECT *,
  company IN ('EQUIFAX, INC.',
              'TRANSUNION INTERMEDIATE HOLDINGS, INC.',
              'Experian Information Solutions Inc.') AS is_bureau,
  (issue ILIKE '%report%' OR issue ILIKE '%credit score%'
   OR issue ILIKE '%investigation%' OR issue ILIKE '%fraud alert%'
   OR issue ILIKE '%identity theft%' OR issue ILIKE '%monitoring%') AS is_credit_report_issue
FROM complaints;
""")

In [34]:
con.execute("""
CREATE OR REPLACE TABLE complaints_scoped AS
SELECT * EXCLUDE (is_bureau, is_credit_report_issue),
  (date_received BETWEEN DATE '2025-01-15' AND DATE '2025-01-20'
   AND company IN ('Block, Inc.', 'Early Warning Services, LLC',
                   'NAVY FEDERAL CREDIT UNION', 'CAPITAL ONE FINANCIAL CORPORATION')
   AND issue IN ('Other transaction problem',
                 'Problem caused by your funds being low',
                 'Managing an account')) AS is_surge_event
FROM complaints_flagged
WHERE NOT is_bureau AND NOT is_credit_report_issue;
""")

In [35]:
con.sql("""
SELECT COUNT(*) AS total,
       SUM(is_bureau::INT) AS bureau_rows,
       SUM(is_credit_report_issue::INT) AS credit_issue_rows,
       SUM((NOT is_bureau AND NOT is_credit_report_issue)::INT) AS kept
FROM complaints_flagged;
""").show()

┌────────┬─────────────┬───────────────────┬────────┐
│ total  │ bureau_rows │ credit_issue_rows │  kept  │
│ int64  │   int128    │      int128       │ int128 │
├────────┼─────────────┼───────────────────┼────────┤
│ 605225 │       52511 │             62834 │ 533015 │
└────────┴─────────────┴───────────────────┴────────┘



In [36]:
con.sql("""
SELECT COUNT(*) AS rows, COUNT(*) FILTER (WHERE is_surge_event) AS surge_rows
FROM complaints_scoped
""").show()

┌────────┬────────────┐
│  rows  │ surge_rows │
│ int64  │   int64    │
├────────┼────────────┤
│ 533015 │      45338 │
└────────┴────────────┘

